In [0]:
from pyspark.sql import functions as F

# Number of rows to generate
num_rows = 5_000_000

# Country list (mix of high-risk and low-risk for the exercise)
countries = ["IR","FR", "IT","NG" ,"DE", "US", "UK", "ES"]

segments = ["Retail", "SME", "Corporate"]

years = [2022, 2023, 2024]
months = list(range(1, 13))

df_exposure = (
    spark.range(0, num_rows)
        .withColumn("loan_id", F.concat(F.lit("L"), F.col("id")))
        .withColumn("client_id", F.concat(F.lit("C"), (F.col("id") % 1_000_000)))
        
        # Random country
        .withColumn("country",
            F.element_at(F.array(*[F.lit(c) for c in countries]),
                         (F.rand() * len(countries)).cast("int") + 1)
        )

        # Random segment
        .withColumn("segment",
            F.element_at(F.array(*[F.lit(s) for s in segments]),
                         (F.rand() * len(segments)).cast("int") + 1)
        )

        # Random exposure amount (EAD)
        .withColumn("ead", (F.rand() * 100_000 + 5_000).cast("double"))

        # Random year and month
        .withColumn("year",
            F.element_at(F.array(*[F.lit(y) for y in years]),
                         (F.rand() * len(years)).cast("int") + 1)
        )
        .withColumn("month",
            F.element_at(F.array(*[F.lit(m) for m in months]),
                         (F.rand() * len(months)).cast("int") + 1)
        )

        .drop("id")
)

display(df_exposure.limit(20))


In [0]:
df_exposure.write.mode("overwrite") \
    .partitionBy("country") \
    .parquet("/mnt/training/loan_exposure")

In [0]:
display(dbutils.fs.ls("/mnt/training/loan_exposure"))


In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

ratings = [
    ("AAA", 0.001, 0.20, "active"),
    ("AA",  0.002, 0.25, "active"),
    ("A",   0.005, 0.30, "active"),
    ("BBB", 0.010, 0.40, "active"),
    ("BB",  0.020, 0.45, "active"),
    ("B",   0.050, 0.60, "inactive")
]

ratings_df = spark.createDataFrame(
    [Row(rating=r, pd=pd, lgd=lgd, status=status) for (r,pd,lgd,status) in ratings]
)

risk_dim = (
    spark.range(0, 200_000)
        .withColumn("loan_id", (col("id") % 1_000_000).cast("string"))
        .withColumn("rating", F.expr("element_at(array('AAA','AA','A','BBB','BB','B'), cast(rand()*6 as int)+1)"))
        .join(ratings_df, "rating")
        .drop("id")
)

risk_dim.write.mode("overwrite").partitionBy("rating").parquet("/mnt/training/risk_parameters")


In [0]:
risk_dim.display()

In [0]:
display(dbutils.fs.ls("/mnt/training/risk_parameters"))


In [0]:
from pyspark.sql import Row

risk_countries = spark.createDataFrame([
    # High-risk (sanctions, AML monitoring)
    Row(country="IR", risk_flag="high"),   # Iran
    Row(country="SY", risk_flag="high"),   # Syria
    Row(country="RU", risk_flag="high"),   # Russia
    Row(country="NG", risk_flag="high"),   # Nigeria
    Row(country="PK", risk_flag="high"),   # Pakistan
    
    # Medium-risk (emerging, financial instability)
    Row(country="BR", risk_flag="medium"), # Brazil
    Row(country="IN", risk_flag="medium"), # India
    Row(country="MX", risk_flag="medium"), # Mexico
    Row(country="TR", risk_flag="medium"), # Türkiye
    
    # Low-risk OECD
    Row(country="FR", risk_flag="low"),
    Row(country="DE", risk_flag="low"),
    Row(country="US", risk_flag="low"),
    Row(country="CA", risk_flag="low"),
    Row(country="UK", risk_flag="low"),
    Row(country="JP", risk_flag="low")
])

display(risk_countries)
risk_countries.write.mode("overwrite").parquet("/mnt/training/risk_countries")



In [0]:
dbutils.fs.rm("/mnt/training/risk_parameters_dim", recurse=True)

In [0]:
dbutils.fs.rm("/mnt/training/risk_parameters", recurse=True)